## Compilation errors

At compilation several error might occur. In this notebook we describe the most common ones, show what are the typical error messages and how to interpret them and explain how to handle them.

Let us start by defining a simple quantum program:

In [ ]:
# Or use built-in graph patterns
from qoolqit import DataGraph, Register

graph = DataGraph.square(m=2, n=2)
register = Register.from_graph(graph)  # 2x2 square lattice
graph.draw()

In [ ]:
from qoolqit import Drive
from qoolqit.waveforms import InterpolatedWaveform, RampWaveform

# Interpolated waveform: smooth curve through specified values
omega_wf = InterpolatedWaveform(duration=50, values=[0, 1, 0])
# Ramp waveform
delta_wf = RampWaveform(duration=50, initial_value=-2, final_value=2)

# Combine into a Drive
drive = Drive(
    amplitude=omega_wf,
    detuning=delta_wf
)

drive.draw()

In [ ]:
from qoolqit import AnalogDevice, QuantumProgram

program = QuantumProgram(
    register=register,
    drive=drive
)

device=AnalogDevice()
program.compile_to(device)

We observe there that the compilation was successful and we can access the compiled sequence.

In [ ]:
program.draw()
program.compiled_sequence.draw()

Let us now explore different cases where different error messages might.
First let us define a small function that output the quantum program we want to compile:


In [ ]:
# Or use built-in graph patterns
from qoolqit import DataGraph, Drive, QuantumProgram, Register
from qoolqit.waveforms import RampWaveform


def create_program(L,amp,det):
    graph = DataGraph.square(m=L, n=L)
    register = Register.from_graph(graph)  # LxL square lattice

    omega_wf = InterpolatedWaveform(duration=50, values=[0, amp, 0])
    delta_wf = RampWaveform(duration=50, initial_value=-det, final_value=det)

    drive = Drive(amplitude=omega_wf, detuning=delta_wf)

    return QuantumProgram(register=register,drive=drive)


## Error 1: Amplitude too large
In this case the compilation routine will try to rescale the register in such a way to maintain the same relative amplitudes of the drive. If the amplitude of the drive is too large, the register may become too large, leading to an overflow error.

In [ ]:
from qoolqit.exceptions import CompilationError

L=4
amp=100
det=2
program=create_program(L,amp,det)
try:
    compiled_sequence = program.compile_to(device)
except CompilationError as err:
    print(err)

>#### What happened?
>We selected an amplitude that is 100 times larger than the maximum atomic pair-wise interaction. The compilation routine automatically attempted to employ the maximum allowed amplitude and rescale the register accordingly: that is in such a way to have the a value of interaction at nearest neighobr that matches the value of the amplitude. In doing so, the register size increased, leading to an overflow error.

## Error 2: detuning too large 

In [ ]:
L=4
amp=1
det=20
program=create_program(L,amp,det)
try:
    compiled_sequence = program.compile_to(device)
except CompilationError as err:
    print(err)

>#### What happened?
>We selected a detuning that is 20 times larger than the maximum atomic pair-wise interaction. The compilation routine automatically rescales the amplitude to the maximum allowed value and all the other quantities accordingly. Hence, it rescaled the register and the detuning. This time the register is fitting the device but the detuning value goes beyond the allowed bounds.

## Error 3: duration too long

In [ ]:
L=2
amp=10
det=1
program=create_program(L,amp,det)
try:
    compiled_sequence = program.compile_to(device)
except CompilationError as err:
    print(err)

>#### What happened?
>The compilation routine automatically rescales the amplitude to the maximum allowed value and all the other quantities accordingly. This means that also the time is rescaled. In particular, as explained in [Adimensionalization and Compilation](../../extended_usage/adimensionalization.md#time-scaling), when the amplitude is too large to fit the device and it has to be decreased and everything rescaled accordingly, then time is increased. In this case the total duration ended up being longer than the allowed value.